# AmsiNet - Step 6: context fusion, prior correction and the 3-way comparison (CPU only)

**Runtime:** Colab **CPU** (no GPU, no compute units to speak of; takes a few minutes). Needs the Step 3a outputs in `MyDrive/amsinet/agents/` (seed-0 probabilities of the page, block, line and word agents) and, for the last section, the Step 4 MDIW-13 zero-shot files.

No network is trained here. Everything is computed from the saved probabilities, so it answers one question quickly:
**does the hierarchy add accuracy over the best single word agent once the levels are fused?**

1. **Prior correction** (logit adjustment): removes the Latin "sink" caused by Latin being about 37% of the training words. The strength tau is chosen on validation only.
2. **Line / block context**: each word also sees the average evidence of the *other* words in its line and block (leave-one-out, so a word never votes for itself).
3. **Cross-level fusion**: log-linear combination of word, line-context, block-context, line agent, block agent and page agent. Weights are chosen on validation (coordinate search); a learned logistic-regression fusion is reported alongside.
4. **Significance**: paired bootstrap over test images (1,000 resamples).
5. **3-way comparison** with the Table 7 setting of the BSTD paper (Hindi vs English vs one regional language).
6. **MDIW-13**: prior correction (tau from BSTD validation) and line context on the zero-shot probabilities.

Decision gate printed at the end of section 4: if the fused system gains less than 1 macro-F1 point with a bootstrap CI that includes 0, the hierarchy does not pay off at word level and we change the story (efficiency) before spending GPU time.

In [ ]:
# 1. Setup (CPU runtime is fine)
from google.colab import drive
import os
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: reconnect and run this cell again'
import json, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
ROOT = '/content/drive/MyDrive/amsinet'
CFG = dict(WORD_TAG='word_moe_seed0',   # which word agent to start from ('word_seed0' or 'word_moe_seed0')
           LINE_TAG='line_seed0', BLOCK_TAG='block_seed0', PAGE_TAG='page_seed0',
           BOOT=1000, SEED=0, EPS=1e-4)

In [ ]:
# 2. Load index, word crops, region table and the saved probabilities
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8'))
SCRIPTS, INDEX = D['scripts'], D['index']; SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS)
IDX = {r['id']: r for r in INDEX}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv'); REGDF = pd.read_csv(f'{ROOT}/data/regions_meta.csv')

def load(tag, sp):
    P = np.load(f'{ROOT}/agents/{tag}_{sp}_probs.npy').astype(np.float32); pa = pd.read_csv(f'{ROOT}/agents/{tag}_{sp}_paths.csv')
    return P, pa.path.tolist()

def region_lookup(level, tag, sp):
    P, paths = load(tag, sp); pos = {p: i for i, p in enumerate(paths)}
    d = REGDF[(REGDF.level == level) & (REGDF.split == sp)]
    return {(r.image_id, r.rid): P[pos[r.path]] for r in d.itertuples() if r.path in pos}

print(len(META), 'word crops |', C, 'scripts:', SCRIPTS)

## 3. Build the per-word feature table (word, line / block context, line / block / page agent)

In [ ]:
def wctx(sp):
    d = META[META.split == sp].reset_index(drop=True)
    Pw, pw = load(CFG['WORD_TAG'], sp); assert pw == d.path.tolist(), 'word probabilities are not aligned with crops_meta.csv'
    n = len(d); logP = np.log(np.clip(Pw, CFG['EPS'], 1))
    lid, bid, wid = [], [], []
    for iid, k in zip(d.image_id, d.word_idx):
        w = IDX[iid]['words'][k]; lid.append(w.get('line', -1)); bid.append(w.get('block', -1))
        b = w['bbox']; wid.append(np.log1p(max(b[2] - b[0], 1)))
    lid, bid, wid = np.array(lid), np.array(bid), np.array(wid)

    def loo(keycol):
        # leave-one-out mean log-probability of the other words that share (image, key)
        key = pd.factorize(d.image_id.astype(str).values + '|' + np.asarray(keycol).astype(str))[0]; ng = key.max() + 1
        S = np.zeros((ng, C)); cnt = np.zeros(ng); np.add.at(S, key, logP); np.add.at(cnt, key, 1)
        other = cnt[key] - 1; F = np.zeros((n, C)); m = other > 0
        F[m] = (S[key][m] - logP[m]) / other[m][:, None]
        return F, other
    Fl, nl = loo(lid); Fb, nb = loo(bid)
    # region agents (missing region -> zero feature = no influence)
    def reg(level, tag, key):
        L = region_lookup(level, tag, sp); F = np.zeros((n, C)); ok = np.zeros(n, bool)
        for i, (iid, kk) in enumerate(zip(d.image_id, key)):
            rid = f'{level[0]}{kk}'
            if (iid, rid) in L: F[i] = np.log(np.clip(L[(iid, rid)], CFG['EPS'], 1)); ok[i] = True
        return F, ok
    La, okl = reg('line', CFG['LINE_TAG'], lid); Ba, okb = reg('block', CFG['BLOCK_TAG'], bid)
    PL = region_lookup('page', CFG['PAGE_TAG'], sp)
    Pa = np.zeros((n, C)); okp = np.zeros(n, bool)
    for i, iid in enumerate(d.image_id):
        if (iid, 'p') in PL: Pa[i] = np.log(np.clip(PL[(iid, 'p')], CFG['EPS'], 1)); okp[i] = True
    y = np.array([SIDX[s] for s in d.script])
    cov = dict(line_ctx=float((nl > 0).mean()), block_ctx=float((nb > 0).mean()), line_agent=float(okl.mean()), block_agent=float(okb.mean()), page_agent=float(okp.mean()))
    return dict(d=d, y=y, word=logP, line_ctx=Fl, block_ctx=Fb, line_agent=La, block_agent=Ba, page_agent=Pa, img=d.image_id.values, cov=cov)

TAB = {sp: wctx(sp) for sp in ('val', 'test')}
for sp in TAB: print(sp, len(TAB[sp]['y']), 'words | coverage:', {k: round(v, 3) for k, v in TAB[sp]['cov'].items()})
TRAIN_PRIOR = np.bincount([SIDX[s] for s in META[META.split == 'train'].script], minlength=C).astype(float); TRAIN_PRIOR /= TRAIN_PRIOR.sum()
print('training prior:', dict(zip(SCRIPTS, TRAIN_PRIOR.round(3))))

## 4. Fusion: prior correction, context, cross-level weights, learned fusion, bootstrap

In [ ]:
def confmat(y, p): return np.bincount(y * C + p, minlength=C * C).reshape(C, C)
def f1_from(cm):
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    den = 2 * tp + fp + fn; return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
def metr(y, p): cm = confmat(y, p); return dict(acc=float(np.trace(cm) / cm.sum()), f1=float(f1_from(cm).mean()))

KEYS = ['line_ctx', 'block_ctx', 'line_agent', 'block_agent', 'page_agent']
def score(T, tau, w):
    s = T['word'] - tau * np.log(TRAIN_PRIOR)[None, :]
    for k in KEYS: s = s + w.get(k, 0.0) * T[k]
    return s
def val_f1(tau, w): return metr(TAB['val']['y'], score(TAB['val'], tau, w).argmax(1))['f1']

GRID_T = [0, .25, .5, .75, 1.0, 1.25]; GRID_W = [0, .1, .25, .5, 1, 1.5, 2, 3]
tau0 = max(GRID_T, key=lambda t: val_f1(t, {}))
print('tau (prior correction only) =', tau0, '| val macro-F1', round(100 * val_f1(tau0, {}), 2), 'vs word agent', round(100 * val_f1(0, {}), 2))
# context only, then all levels, by coordinate search on validation
def coord(keys, tau_free=True, passes=2):
    w = {k: 0.0 for k in keys}; tau = tau0
    for _ in range(passes):
        for k in keys: w[k] = max(GRID_W, key=lambda g: val_f1(tau, {**w, k: g}))
        if tau_free: tau = max(GRID_T, key=lambda t: val_f1(t, w))
    return tau, w
tau_c, w_c = coord(['line_ctx', 'block_ctx'])
tau_f, w_f = coord(KEYS)
print('context only: tau', tau_c, w_c); print('all levels  : tau', tau_f, w_f)

# learned fusion (multinomial LR on the stacked log-probabilities, fitted on validation only)
def stackX(T): return np.hstack([T['word']] + [T[k] for k in KEYS])
LR = LogisticRegression(max_iter=3000, C=0.5).fit(stackX(TAB['val']), TAB['val']['y'])
print('LR classes seen:', len(LR.classes_), 'of', C)
def lr_pred(T):
    pr = LR.predict_proba(stackX(T)); full = np.zeros((len(pr), C)); full[:, LR.classes_] = pr; return full.argmax(1)

T = TAB['test']; y = T['y']
PRED = {'word agent': score(T, 0, {}).argmax(1),
        '+ prior correction': score(T, tau0, {}).argmax(1),
        '+ line/block context': score(T, tau_c, w_c).argmax(1),
        '+ all levels (weights on val)': score(T, tau_f, w_f).argmax(1),
        'learned fusion (LR on val)': lr_pred(T)}
rows = []
for k, p in PRED.items():
    m = metr(y, p); rows.append((k, round(100 * m['acc'], 2), round(100 * m['f1'], 2)))
FUS = pd.DataFrame(rows, columns=['system (BSTD test words)', 'accuracy %', 'macro-F1 %']); display(FUS)

In [ ]:
# paired bootstrap over test images
rng = np.random.RandomState(CFG['SEED']); imgs, inv = np.unique(T['img'], return_inverse=True); ni = len(imgs)
def per_img_cm(p):
    M = np.zeros((ni, C * C)); np.add.at(M, (inv, y * C + p), 1); return M
CM = {k: per_img_cm(p) for k, p in PRED.items()}
cnt = rng.multinomial(ni, np.ones(ni) / ni, size=CFG['BOOT']).astype(float)
def boot(k):
    cms = (cnt @ CM[k]).reshape(-1, C, C)
    acc = np.trace(cms, axis1=1, axis2=2) / cms.sum((1, 2))
    tp = np.diagonal(cms, axis1=1, axis2=2); fp = cms.sum(1) - tp; fn = cms.sum(2) - tp
    den = 2 * tp + fp + fn; f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0).mean(1); return acc, f1
base_acc, base_f1 = boot('word agent'); rows = []
for k in PRED:
    if k == 'word agent': continue
    a, f = boot(k); da, df = a - base_acc, f - base_f1
    rows.append((k, f'{100 * da.mean():+.2f} [{100 * np.percentile(da, 2.5):+.2f}, {100 * np.percentile(da, 97.5):+.2f}]',
                    f'{100 * df.mean():+.2f} [{100 * np.percentile(df, 2.5):+.2f}, {100 * np.percentile(df, 97.5):+.2f}]'))
GAIN = pd.DataFrame(rows, columns=['vs word agent', 'delta accuracy (pts) [95% CI]', 'delta macro-F1 (pts) [95% CI]']); display(GAIN)
lo = np.percentile(boot('+ all levels (weights on val)')[1] - base_f1, 2.5)
mf = metr(y, PRED['+ all levels (weights on val)'])['f1'] - metr(y, PRED['word agent'])['f1']
print('\nDECISION GATE: fused macro-F1 gain = %+.2f pts, CI lower bound %+.2f ->' % (100 * mf, 100 * lo),
      'CONTEXT PAYS OFF (keep the hierarchy story)' if (mf >= 0.01 and lo > 0) else 'WEAK / NOT SIGNIFICANT (tell me; we change the angle before spending GPU)')

In [ ]:
# per-script F1, Latin-sink check, how often words of one line disagree
cmw, cmf = confmat(y, PRED['word agent']), confmat(y, PRED['+ all levels (weights on val)'])
PS = pd.DataFrame({'script': SCRIPTS, 'support': cmw.sum(1), 'F1 word agent': (100 * f1_from(cmw)).round(1), 'F1 fused': (100 * f1_from(cmf)).round(1)})
PS['delta'] = (PS['F1 fused'] - PS['F1 word agent']).round(1); display(PS)
li = SIDX['Latin']
for k in ('word agent', '+ prior correction', '+ all levels (weights on val)'):
    nl = y != li; print(f'{k:34s} non-Latin words predicted Latin: {100 * (PRED[k][nl] == li).mean():.2f}%')
d = T['d'].copy(); d['pred'] = PRED['word agent']
d['line'] = [IDX[i]['words'][k].get('line', -1) for i, k in zip(d.image_id, d.word_idx)]
g = d.groupby(['image_id', 'line']).script.nunique(); print('lines with more than one true script: %.1f%% of %d lines' % (100 * (g > 1).mean(), len(g)))
PS.to_csv(f'{ROOT}/results/context_per_script.csv', index=False); FUS.to_csv(f"{ROOT}/results/context_fusion_table_{CFG['WORD_TAG']}.csv", index=False); GAIN.to_csv(f'{ROOT}/results/context_fusion_gain.csv', index=False)

## 5. The 3-way setting of the BSTD paper (Table 7): Hindi vs English vs one regional language

The paper restricts the classifier to {Hindi, English, regional language X}. We take our probabilities (or fused scores), keep the three matching classes and renormalise. Caveats written into the table: our `Devanagari` class merges Hindi and Marathi, so Marathi and Bengali/Assamese rows cannot be reproduced; our test words are the official image split, not the paper's balanced sample. We report the **balanced accuracy** (mean per-class recall over the three classes), which is the closest equivalent to a balanced draw, and the plain accuracy.

In [ ]:
PAPER = {'Kannada': 94.5, 'Malayalam': 90.8, 'Odia': 93.5, 'Tamil': 93.6, 'Telugu': 95.0, 'Gujarati': 83.1, 'Gurmukhi': 92.3}   # Table 7, ViT-B ('Proposed'); Punjabi = Gurmukhi
SC = {'word agent': score(T, 0, {}), 'fused (all levels)': score(T, tau_f, w_f)}
dev, lat = SIDX['Devanagari'], SIDX['Latin']; rows = []
for reg_s, pap in PAPER.items():
    r = SIDX[reg_s]; sel = np.isin(y, [dev, lat, r]); yy = y[sel]; row = {'regional': reg_s, 'test words (3 classes)': int(sel.sum())}
    for name, S in SC.items():
        s3 = S[sel][:, [dev, lat, r]]; p3 = np.array([dev, lat, r])[s3.argmax(1)]
        rec = [(p3[yy == c] == c).mean() for c in (dev, lat, r)]
        row[f'{name} acc %'] = round(100 * (p3 == yy).mean(), 1); row[f'{name} balanced acc %'] = round(100 * float(np.mean(rec)), 1)
    row['paper ViT-B %'] = pap; rows.append(row)
THREE = pd.DataFrame(rows); display(THREE); THREE.to_csv(f'{ROOT}/results/three_way_vs_paper.csv', index=False)

## 6. MDIW-13: prior correction and line context on the zero-shot probabilities

In [ ]:
import glob
tagd = 'printed'
W = pd.read_csv(f'{ROOT}/mdiw/zeroshot_{tagd}_paths.csv'); files = sorted(glob.glob(f'{ROOT}/mdiw/zeroshot_*_{tagd}_probs.npy')); print([os.path.basename(f) for f in files])
f = [x for x in files if f"zeroshot_{CFG['WORD_TAG']}_{tagd}_probs.npy" == os.path.basename(x)]
if not f:
    print('No MDIW-13 zero-shot file for', CFG['WORD_TAG'], '-> skipping this section (zero-shot files exist only for word_seed0 / word_moe_seed0; run Step 4 with this agent if you need them).')
else:
    f = f[0]; print('using', os.path.basename(f))
    Pm = np.load(f).astype(np.float32); assert len(Pm) == len(W)
    ym = np.array([SIDX[s] for s in W.script]); lg = np.log(np.clip(Pm, CFG['EPS'], 1))
    def loo_m(lg, key):
        k = pd.factorize(W[key])[0]; S = np.zeros((k.max() + 1, C)); n = np.zeros(k.max() + 1); np.add.at(S, k, lg); np.add.at(n, k, 1)
        o = n[k] - 1; F = np.zeros_like(lg); m = o > 0; F[m] = (S[k][m] - lg[m]) / o[m][:, None]; return F
    Fl = loo_m(lg, 'line'); Fd = loo_m(lg, 'doc')
    rows = []
    for name, s in [('word agent (zero-shot)', lg), ('+ prior correction (tau from BSTD val = %.2f)' % tau0, lg - tau0 * np.log(TRAIN_PRIOR)),
                    ('+ line context', lg - tau0 * np.log(TRAIN_PRIOR) + 1.0 * Fl), ('+ line + document context', lg - tau0 * np.log(TRAIN_PRIOR) + 1.0 * Fl + 1.0 * Fd)]:
        for sp in ('all', 'test'):
            sel = np.ones(len(W), bool) if sp == 'all' else (W.split == 'test').values
            m = metr(ym[sel], s[sel].argmax(1)); rows.append((name, sp, int(sel.sum()), round(100 * m['acc'], 2), round(100 * m['f1'], 2)))
    MD = pd.DataFrame(rows, columns=['system', 'words', 'n', 'accuracy %', 'macro-F1 %']); display(MD); MD.to_csv(f'{ROOT}/results/mdiw_prior_context.csv', index=False)
    print('Note: every MDIW-13 line and document holds one script, so context gain here is an upper bound that tests consistency, not mixed-script routing.')